# Visualization 01 Streamgraph

Standalone, portable notebook a Top N streamgraph of name popularity over time sliding window, Raw/Share, sex. Renamed from sketch 1.5. The shared setup
imports, data loading is included
below; chart data is inlined so the HTML renders on Linux, macOS and Windows.
Running it writes the executed `_out.ipynb` and `Visualization 01 - Streamgraph.png`.

## Shared setup

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF CONTAINED and renders on
# all platforms incl. macOS: external altair data *.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches 1.13 / 2.4 / 2.6 restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory lean load: another heavy job is running on this machine ~2.5 GB free,
# so we read the CSV with categorical dtypes 15k name strings stored once, derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects the strings stay shared so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all years aggregate 2.4.
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## Name universe restriction

In [ ]:
# Portability + memory: keep the INLINED data small by limiting the name
# universe to names that ever reached the yearly top 60, always unioned with this sketch's own
# default names so the default view is unchanged.
import gc
_EXTRA = [x for x in [] if x in set(named['preusuel'])]
_yr = (named.dropna(subset=['year'])
       .groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
_yr['rk'] = _yr.groupby('year')['nombre'].rank(ascending=False, method='first')
_universe = set(_yr.loc[_yr['rk'] <= 60, 'preusuel'])
del _yr
POPULAR_NAMES = sorted(_universe.union(_EXTRA))
named = named[named['preusuel'].isin(POPULAR_NAMES)].copy()
ALL_NAMES = POPULAR_NAMES
gc.collect()
print(f'Universe: {len(POPULAR_NAMES):,} names (ever reached the yearly top 60 + defaults); {len(named):,} rows kept.')

## The sketch

### Sketch 1.5 "Volume" streamgraph

Centred stacked bands of the Top N names. Pick a window size once, then slide
its start across the century with the second slider: the window keeps a constant
width, which makes it easy to compare equal length periods and to zoom past the
post war birth volume bump without re tuning two handles. The Top N is ranked on the
visible window; a sex dropdown filters Mixed/Boys/Girls; click a band to isolate
it. Y axis = number of births, not a share.

In [ ]:
# Full year, name, sex mode series. The Top N ranking is computed client side
# on the selected window, so every name must be available to the chart.
_stream_base = named.dropna(subset=['year'])
stream_mixte = (_stream_base.groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
stream_mixte['sex_mode'] = 'Mixed'
stream_sex = (_stream_base.groupby(['year', 'preusuel', 'sexe'], as_index=False)['nombre'].sum())
stream_sex['sex_mode'] = np.where(stream_sex['sexe'].eq(1), 'Boys', 'Girls')
stream_src = pd.concat([
    stream_mixte[['year', 'preusuel', 'nombre', 'sex_mode']],
    stream_sex[['year', 'preusuel', 'nombre', 'sex_mode']],
], ignore_index=True)
stream_src['year'] = stream_src['year'].astype(int)

# Fixed size SLIDING window: set the width once, then slide the start. The window
# is [win_start, win_start + win_size]; default 120y from 1900 == the full span.
win_size = alt.param(name='win_size', value=120,
                     bind=alt.binding_range(min=5, max=120, step=5,
                                            name='Window size years '))
win_start = alt.param(name='win_start', value=1900,
                      bind=alt.binding_range(min=1900, max=2020, step=1,
                                             name='Window start slide '))
n_top = alt.param(name='n_top', value=8,
                  bind=alt.binding_range(min=3, max=50, step=1,
                                         name='Top N names ranked on the window '))
sex_15 = alt.param(name='sex_15', value='Mixed',
                   bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                           name='Sex '))
# Raw births vs Share of the top N: 'Share' divides each name by the top N total
# of its year, so the stacked height stays ~constant and recent decades are no
# longer crushed by the post war birth volume bump a recurring review request.
scale_15 = alt.param(name='scale_15', value='abs',
                     bind=alt.binding_select(options=['abs', 'rel'],
                                             labels=['absolute births',
                                                     'relative share of top N'],
                                             name='Scale '))

isolate = alt.selection_point(fields=['preusuel'], on='click', clear='dblclick', empty=True)

# Shared pipeline: sex filter > window filter > per name total on the window >
# dense rank > keep the Top N > per name in window peak for the labels.
def piped(chart):
    return (chart
            .transform_filter('datum.sex_mode == sex_15')
            .transform_filter('datum.year >= win_start && datum.year <= win_start + win_size')
            .transform_joinaggregate(tot='sum(nombre)', groupby=['preusuel'])
            .transform_window(rk='dense_rank()',
                              sort=[alt.SortField('tot', order='descending')])
            .transform_filter('datum.rk <= n_top')
            .transform_joinaggregate(maxn='max(nombre)', groupby=['preusuel'])
            .transform_joinaggregate(ytot='sum(nombre)', groupby=['year'])
            .transform_calculate(
                yval="scale_15 == 'rel' ? datum.nombre / datum.ytot : datum.nombre"))

bands = piped(alt.Chart(stream_src)).mark_area(interpolate='monotone').encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(format='d', grid=False)),
    y=alt.Y('yval:Q', stack='center', title=None, axis=None),
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
    order=alt.Order('tot:Q', sort='descending'),
    opacity=alt.condition(isolate, alt.value(0.9), alt.value(0.18)),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'),
             alt.Tooltip('sex_mode:N', title='Sex mode'),
             alt.Tooltip('year:Q', title='Year', format='d'),
             alt.Tooltip('nombre:Q', title='Births', format=',')],
)
# Label layers share the SAME stacked pipeline so they land on the bands and only
# print text on each band's in window peak row.
label_text = alt.condition('datum.nombre == datum.maxn',
                           alt.Text('preusuel:N'), alt.value(''))
halo = piped(alt.Chart(stream_src)).mark_text(
    baseline='top', dy=2, fontSize=11, fontWeight='bold',
    stroke='white', strokeWidth=3).encode(
    x='year:Q', y=alt.Y('yval:Q', stack='center', axis=None),
    detail='preusuel:N', order=alt.Order('tot:Q', sort='descending'), text=label_text)
labels = piped(alt.Chart(stream_src)).mark_text(
    baseline='top', dy=2, fontSize=11, fontWeight='bold', color='#111').encode(
    x='year:Q', y=alt.Y('yval:Q', stack='center', axis=None),
    detail='preusuel:N', order=alt.Order('tot:Q', sort='descending'), text=label_text)

sketch_1_5 = (bands + halo + labels).add_params(isolate, win_size, win_start, n_top, sex_15, scale_15).properties(
    width=820, height=380,
    title='Visualization 01 Streamgraph of the Top N names')

sketch_1_5.save('Visualization 01 - Streamgraph.png', ppi=120)
sketch_1_5
